# 3.14 - Debogage d un entrainement qui tourne mais n apprend pas
**Navigation** : [<< 3.13b - Deux vraies cartes : DDP et FSDP sous nccl](3.13b-Deux-Cartes-Reelles-DDP-FSDP-nccl.ipynb) · [Feuille de route de la serie](README.md)

Carnet pedagogique du parcours 03-Deep-Learning. Il vise a **diagnostiquer** un modele qui boucle sans faire baisser la perte : loss constante, loss qui oscille, loss qui explose, loss qui descend mais plafonne.

**Cinq controles** systematiques dans l ordre :
1. **Data pipeline** - X.shape, range, normalisation, NaN/Inf dans y.
2. **Loss correcte** - perte adaptee a la tache (regression / binaire / multi-classes).
3. **Gradient** - norme non-NaN, valeurs raisonnables (entre 0 et quelques dizaines).
4. **Learning rate** - ni trop petit (loss stagne), ni trop grand (loss oscille ou explose).
5. **Initialisation** - poids pas tous a zero, variance raisonnable.

Ce carnet utilise **`SGDClassifier`** de scikit-learn (CPU strict) pour montrer ces patterns sur des donnees jouet (classification binaire). La perte est mesuree explicitement par `log_loss(y, clf.predict_proba(X))` apres chaque passe -- `SGDClassifier` n expose pas de `loss_curve_`.

In [1]:
# %pip install -q scikit-learn numpy

import numpy as np
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import log_loss

rng = np.random.default_rng(seed=42)
n = 400
X = rng.normal(size=(n, 2))
y = (X[:, 0] + X[:, 1] > 0).astype(int)
print(f"X.shape={X.shape}, y.shape={y.shape}, classes={np.unique(y).tolist()}")
print(f"X min/max: {X.min():.2f} / {X.max():.2f}")
print(f"NaN dans X: {np.isnan(X).sum()}, NaN dans y: {np.isnan(y).sum()}")

X.shape=(400, 2), y.shape=(400,), classes=[0, 1]
X min/max: -2.96 / 2.91
NaN dans X: 0, NaN dans y: 0


## 1. Quand la perte stagne - lr trop petit

**Symptome** - la perte ne bouge pas d une epoch a l autre (ou descend si lentement que le budget d epochs est insuffisant).

**Cause typique** - `learning_rate` initial trop petit (`eta0=1e-6`) ou `learning_rate="optimal"` avec une regularisation trop forte.

In [2]:
# Pattern 1 - loss stagne avec eta0 = 1e-6
clf = SGDClassifier(
    loss="log_loss",
    learning_rate="constant",
    eta0=1e-6,
    max_iter=50,
    tol=None,
    random_state=42,
)
losses_stagne = []
for _ in range(50):
    clf.partial_fit(X, y, classes=[0, 1])
    losses_stagne.append(log_loss(y, clf.predict_proba(X)))

print(f"loss initiale : {losses_stagne[0]:.6f}")
print(f"loss finale   : {losses_stagne[-1]:.6f}")
print(f"evolution     : {losses_stagne[0] - losses_stagne[-1]:.2e}")

sample = [losses_stagne[i] for i in (0, 10, 20, 30, 40, 49)]
for i, e in zip([0, 10, 20, 30, 40, 49], sample):
    print(f"  epoch {i:2d} - loss = {e:.6f}")

loss initiale : 0.693075
loss finale   : 0.689577
evolution     : 3.50e-03
  epoch  0 - loss = 0.693075
  epoch 10 - loss = 0.692358
  epoch 20 - loss = 0.691643
  epoch 30 - loss = 0.690929
  epoch 40 - loss = 0.690217
  epoch 49 - loss = 0.689577


**Lecture du resultat.** La perte va de 0.693075 a 0.689577 en 50 passes - l evolution totale est de 3.50e-03. Elle reste collee au **niveau du hasard** (ln 2 = 0.693, le pile-ou-face) : la courbe descend de facon monotone, reguliere, mais elle ne converge pas, et le modele n apprend **rien**.

Les six points affiches le disent : 0.693075 au depart, 0.692358 a la 10e passe, 0.691643 a la 20e, 0.690929 a la 30e, 0.690217 a la 40e, 0.689577 a la 49e. La pente est constante mais minuscule - le modele se deplace, mais de maniere infime. C est la forme exacte d un entrainement **sain mais trop lent** : rien ne signale une erreur, la courbe ne fait seulement que ne pas descendre.

**Diagnostic.** Premier reflexe - verifier le **learning rate**. Ici `eta0=1e-6` est 4 ordres de grandeur en dessous du lr de reference (0.01) : a cette echelle, les gradients ne poussent pas les poids assez pour que la perte bouge. Un second reflexe, plus loin dans ce carnet, sera le **check 3** : c est lui qui distingue un pas trop petit d un gradient qui a disparu.

In [3]:
# Exercice 1 - reproduire le pattern loss stagne avec un modele lineaire simple.
#
# Objectif - trouver un learning_rate et un nombre d epochs tels que la perte
# reste quasi constante (< 1e-4 de variation) sur SGDClassifier.
#
# Indices -
# - learning_rate="constant", eta0 petit (10^-7, 10^-8)
# - max_iter=20-30
# - tol=None
# - affichez loss initiale, loss finale, et l evolution

pass  # Remplacez par votre code (regle C.1 - pas de raise, juste pass / print)

## 2. Quand la perte oscille ou explose - lr trop grand

**Symptome** - la perte saute d une passe a l autre sans se stabiliser (oscillation), ou depasse 1 : le modele devient confiant dans ses erreurs.

**Cause typique** - `learning_rate` initial trop grand (`eta0=10.0` et au-dela) : chaque pas de gradient franchit la vallee, les poids passent d un cote puis de l autre, et la perte ne s installe jamais.

In [4]:
# Pattern 2 - loss instable avec eta0 = 10.0 (oscille sans converger)
clf2 = SGDClassifier(
    loss="log_loss",
    learning_rate="constant",
    eta0=10.0,
    max_iter=10,
    tol=None,
    random_state=42,
)
losses_explose = []
for _ in range(10):
    try:
        clf2.partial_fit(X, y, classes=[0, 1])
        losses_explose.append(log_loss(y, clf2.predict_proba(X)))
    except Exception as e:
        losses_explose.append(float("nan"))

print(f"loss epoch 0 - {losses_explose[0]:.4f}")
print(f"loss epoch 5 - {losses_explose[5]:.4f}")
print(f"loss epoch 9 - {losses_explose[9]:.4f}")

import math
finite = [l for l in losses_explose if math.isfinite(l)]
print(f"epochs finis - {len(finite)} / {len(losses_explose)}")
if finite:
    print(f"loss max (finie) - {max(finite):.4e}")

loss epoch 0 - 0.6612
loss epoch 5 - 0.2422
loss epoch 9 - 0.5280
epochs finis - 10 / 10
loss max (finie) - 7.4906e-01


**Lecture du resultat.** La perte oscille - 0.6612 au depart, 0.2422 a la 5e passe, 0.5280 a la 9e - sans jamais se stabiliser : ce n est pas une convergence plus lente, c est une **trajectoire instable**. La perte maximale sur la trajectoire reste d ailleurs finie (7.4906e-01) : le modele n explose pas, il **rebondit**. Le journal indique 10 / 10 epochs termines : l oscillation est visible des les premieres passes, elle ne s amortit pas.

Un entrainement qui oscille autour d un plancher sans jamais le franchir est un symptome de **pas trop grand**, pas de donnees fausses - le diagnostic du §4 le confirme en marchant sur la valeur de `eta0`.

**Diagnostic.** Premier reflexe - reduire `eta0` d un facteur 10 a 1000. L alternative `penalty=elasticnet` ou `learning_rate="adaptive"` (sklearn) adoucit la trajectoire. Cote reseau, le meme symptome porte un nom dans l instrument du §5 : c est le **check 4**, mesure sur la trajectoire elle-meme.

In [5]:
# Exercice 2 - reproduire le pattern loss instable et trouver le seuil d instabilite.
#
# Objectif - pour eta0 variant entre 0.1 et 50.0, observer a partir de quel eta0
# la perte devient instable sur SGDClassifier(log_loss, learning_rate="constant") :
# oscillations entre passes (saut > 0.1 entre deux epochs consecutives) ou
# perte finale > 1 (le modele se trompe avec confiance).
#
# Sortie attendue - un tableau textuel eta0 vers perte_finale.

pass  # Remplacez par votre code

## 3. Quand la perte descend bien - lr bien calibre

**Pattern de reference**. Avec `eta0=0.01`, la perte quitte le niveau du hasard des la premiere passe (0.69 -> 0.38), puis descend jusqu a environ 0.11 en 30 passes.

In [6]:
# Pattern 3 - loss descend bien
clf3 = SGDClassifier(
    loss="log_loss",
    learning_rate="constant",
    eta0=0.01,
    max_iter=30,
    tol=None,
    random_state=42,
)
losses_ok = []
for _ in range(30):
    clf3.partial_fit(X, y, classes=[0, 1])
    losses_ok.append(log_loss(y, clf3.predict_proba(X)))

print(f"loss initiale - {losses_ok[0]:.4f}")
print(f"loss finale   - {losses_ok[-1]:.4f}")
print(f"evolution     - {losses_ok[0] - losses_ok[-1]:+.4f}")

print()
print("Comparaison 3 patterns -")
print(f"  Pattern 1 (lr trop petit)   - loss finale = {losses_stagne[-1]:.4f}")
print(f"  Pattern 2 (lr trop grand)   - loss finale = {losses_explose[-1]:.4e}")
print(f"  Pattern 3 (lr bien calibre) - loss finale = {losses_ok[-1]:.4f}")

loss initiale - 0.3788
loss finale   - 0.1107
evolution     - +0.2681

Comparaison 3 patterns -
  Pattern 1 (lr trop petit)   - loss finale = 0.6896
  Pattern 2 (lr trop grand)   - loss finale = 5.2797e-01
  Pattern 3 (lr bien calibre) - loss finale = 0.1107


**Lecture du resultat.** Des la premiere passe la perte tombe a 0.3788, puis elle descend jusqu a 0.1107 apres 30 passes - l evolution est de -0.2681, contre 3.50e-03 pour le pattern 1. Le modele apprend.

Le bloc de comparaison tranche les trois patterns d un coup : pattern 1 (lr trop petit) 0.6896, pattern 2 (lr trop grand) 5.2797e-01, pattern 3 (lr bien calibre) 0.1107. Ces trois lignes sont la **grille de reference** du carnet : tout diagnostic qui produit un nombre hors de cette bande est suspect, et c est cette bande qui rendra ses verdicts au §5.

**Pourquoi `eta0=0.01` marche ?** L echelle du learning rate doit correspondre a l echelle des gradients (lui-meme dependant de l echelle des features). Avec des features en `~N(0, 1)`, eta0=0.01 est dans la bonne fenetre.

## 4. Diagnostic en 5 checks

Quand l entrainement ne baisse pas, executer ces 5 checks **dans l ordre** (du plus rapide au plus lent) -

| # | Question | Verification |
|---|---|---|
| 1 | Data pipeline | X.shape, range, normalisation, NaN/Inf dans y |
| 2 | Loss correcte | perte adaptee a la tache (pas de MSE sur classification multi-classes) |
| 3 | Gradient | norme non-NaN, valeurs raisonnables (entre 0 et quelques dizaines) |
| 4 | Learning rate | ni trop petit (stagne), ni trop grand (oscille ou explose) |
| 5 | Initialisation | poids pas tous a zero, variance raisonnable |

Mini-outil ci-dessous implemente les checks 1 et 4 (les 3 autres dependent du framework).

In [7]:
def diagnostic_data(X, y):
    """Check 1 - data pipeline."""
    msgs = []
    if np.isnan(X).any():
        msgs.append(f"KO - X contient {np.isnan(X).sum()} NaN")
    if np.isnan(y).any():
        msgs.append(f"KO - y contient {np.isnan(y).sum()} NaN")
    if np.isinf(X).any() or np.isinf(y).any():
        msgs.append("KO - X ou y a des Inf")
    if X.std() < 1e-6 or X.std() > 100:
        msgs.append(f"WARN - X.std()={X.std():.4f} (range attendu ~1)")
    if not msgs:
        msgs.append(f"OK - X.shape={X.shape}, X.std={X.std():.3f}, NaN=0")
    return msgs

def diagnostic_lr(eta0, n_iter):
    """Check 4 - learning rate sur SGDClassifier(log_loss)."""
    clf = SGDClassifier(loss="log_loss", learning_rate="constant", eta0=eta0,
                        max_iter=n_iter, tol=None, random_state=42)
    losses = []
    for _ in range(n_iter):
        clf.partial_fit(X, y, classes=[0, 1])
        losses.append(log_loss(y, clf.predict_proba(X)))
    fin = losses[-1]
    saut_max = max(abs(b - a) for a, b in zip(losses, losses[1:]))
    if fin > 1.0:
        return [f"KO - lr={eta0} -> explose (perte finale {fin:.2f} > 1 : le modele se trompe avec confiance)"]
    if saut_max > 0.1:
        return [f"KO - lr={eta0} -> oscille sans converger (saut max {saut_max:.2f} entre deux passes)"]
    if fin > 0.68:
        return [f"KO - lr={eta0} -> stagne (perte finale {fin:.4f}, au voisinage du hasard ln(2) = 0.693)"]
    return [f"OK - lr={eta0} -> converge (perte finale {fin:.4f})"]

print("=== Diagnostic data ===")
for msg in diagnostic_data(X, y):
    print(f"  {msg}")

print()
print("=== Diagnostic lr ===")
for eta in [1e-6, 0.01, 0.1, 1.0, 10.0, 50.0]:
    for msg in diagnostic_lr(eta, n_iter=30):
        print(f"  {msg}")

=== Diagnostic data ===
  OK - X.shape=(400, 2), X.std=0.984, NaN=0

=== Diagnostic lr ===


  KO - lr=1e-06 -> stagne (perte finale 0.6910, au voisinage du hasard ln(2) = 0.693)


  OK - lr=0.01 -> converge (perte finale 0.1107)


  OK - lr=0.1 -> converge (perte finale 0.0548)


  OK - lr=1.0 -> converge (perte finale 0.0357)


  KO - lr=10.0 -> oscille sans converger (saut max 0.54 entre deux passes)


  KO - lr=50.0 -> explose (perte finale 3.31 > 1 : le modele se trompe avec confiance)


**Lecture du resultat.** L instrument lineaire rend deux blocs.

- **Data** - `OK - X.shape=(400, 2), X.std=0.984, NaN=0` : pipeline OK, features standardisees, aucun NaN ni Inf.
- **lr** - `1e-06` stagne (perte finale 0.6910, au voisinage du hasard ln(2) = 0.693) ; `0.01` converge (0.1107), `0.1` converge (0.0548), `1.0` converge (0.0357) ; `10.0` oscille sans converger (saut max 0.54 entre deux passes) ; `50.0` explose (perte finale 3.31 > 1 : le modele se trompe avec confiance).

**Ce que la grille etablit.** La fenetre viable se situe **entre `0.01` et `1.0`** : elle **resserre** la borne basse entre `1e-06` et `0.01` sans la resserrer plus finement, et elle montre que la borne haute est franchie bien avant `50.0` - a `10.0` la trajectoire oscille deja. Le diagnostiqueur tranche donc les six taux testes : stagne / converge / oscille / explose.

**Ce que cet instrument ne peut pas faire.** Sur un modele lineaire, il n y a qu un jeu de poids et une seule echelle de valeurs. Les checks **2** (loss adaptee a la tache), **3** (gradient) et **5** (initialisation) n ont rien a observer : ni matrice de poids a inspecter, ni gradient a propager, ni distribution d initialisation a comparer. Les trois controles manquants demandent un **reseau** - c est l objet de la section suivante.

In [8]:
# Exercice 3 - appliquer le diagnostic a un modele malade.
#
# Objectif - fabriquez vous-meme une configuration dont la perte stagne ou explose (par exemple
# un taux d'apprentissage hors de la fenetre viable mesuree plus haut).
# A vous d utiliser diagnostic_data() et diagnostic_lr() pour identifier la cause.
#
# Sortie attendue - pour chaque lr teste, le verdict (OK / KO) et la cause probable.

# Votre code ici (regle C.1 - pas de raise)
pass

## 5. Les trois controles qui ne passent pas sur un modele lineaire

Les checks 1 (data) et 4 (learning rate) s instrumentent sur `SGDClassifier` : une seule matrice de poids, une seule echelle de valeurs, et le diagnostic tient en moins de cinq minutes. Les checks 2 (loss adaptee), 3 (gradient) et 5 (initialisation) demandent, eux, un **reseau** : plusieurs couches de poids, un gradient qui se propage de proche en proche, une initialisation dont la distribution compte.

Cette section branche donc un petit **MLP PyTorch (CPU strict)** sur les memes donnees et outille les trois controles manquants. Le reseau est volontairement minuscule - 2 entrees, 16 neurones caches, tanh, SGD - pour que chaque chiffre reste lisible. Les lectures qui suivent parcourent, dans l ordre : le fonctionnement d un reseau **sain**, la panne d **initialisation**, l affaiblissement du **gradient**, et enfin la **loss** qui affiche un nombre rassurant pour une metrique degradee. Une derniere cellule rassemble les cinq controles dans un instrument reutilisable, applique a quatre configurations.

La reproductibilite est assuree : `torch.set_num_threads(1)` fixe l ordre de reduction CPU (il depend du nombre de threads), et toutes les graines sont posees explicitement, de sorte que les valeurs affichees sont les memes d une machine a l autre.

In [9]:
# Le reseau qui rend visibles les checks 2, 3 et 5 : un MLP PyTorch, CPU strict.
import torch
import torch.nn as nn

torch.set_num_threads(1)   # l ordre de reduction CPU depend du nombre de threads
torch.manual_seed(0)

X_t = torch.tensor(X, dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)


def mlp(hidden=16, depth=2, init="kaiming"):
    """MLP tanh a `depth` couches cachees. init='zeros' casse l initialisation."""
    torch.manual_seed(0)
    couches, entrees = [], 2
    for _ in range(depth):
        couches += [nn.Linear(entrees, hidden), nn.Tanh()]
        entrees = hidden
    couches.append(nn.Linear(entrees, 1))
    modele = nn.Sequential(*couches)
    if init == "zeros":
        for p in modele.parameters():
            nn.init.zeros_(p)
    return modele


def entrainer(modele, perte, pas=200, lr=0.05):
    """SGD full-batch. Renvoie (historique de perte, normes de gradient du 1er pas)."""
    opt = torch.optim.SGD(modele.parameters(), lr=lr)
    pertes, grads = [], {}
    for i in range(pas):
        opt.zero_grad()
        p = perte(modele(X_t), y_t)
        p.backward()
        if i == 0:
            grads = {n: float(q.grad.norm())
                     for n, q in modele.named_parameters() if q.grad is not None}
        opt.step()
        pertes.append(float(p))
    return pertes, grads


def vraies_metriques(modele):
    """La metrique de la tache (log_loss) et l exactitude, calculees sur les probabilites."""
    z = modele(X_t).detach().numpy().ravel()
    return float(log_loss(y, 1.0 / (1.0 + np.exp(-z)))), float(((z > 0).astype(int) == y).mean())


ref = mlp()
nb_param = sum(p.numel() for p in ref.parameters())
pertes_ref, grads_ref = entrainer(ref, nn.BCEWithLogitsLoss(), pas=200, lr=0.05)
ll_ref, acc_ref = vraies_metriques(ref)
print("=== Reseau de reference (MLP tanh, CPU) ===")
print(f"  architecture        : 2 couches cachees de 16 neurones (tanh), sortie lineaire 1")
print(f"  parametres          : {nb_param}")
print(f"  perte affichee      : {pertes_ref[0]:.6f} -> {pertes_ref[-1]:.6f} (200 pas, lr=0.05)")
print(f"  log_loss (tache)    : {ll_ref:.6f}")
print(f"  exactitude          : {acc_ref:.4f} ({int(round(acc_ref * len(y)))}/400)")
print("  gradient du 1er pas :")
for n, v in grads_ref.items():
    print(f"    {n:12s} : {v:.6e}")

=== Reseau de reference (MLP tanh, CPU) ===
  architecture        : 2 couches cachees de 16 neurones (tanh), sortie lineaire 1
  parametres          : 337
  perte affichee      : 0.702464 -> 0.108086 (200 pas, lr=0.05)
  log_loss (tache)    : 0.107626
  exactitude          : 0.9900 (396/400)
  gradient du 1er pas :
    0.weight     : 6.563111e-02
    0.bias       : 1.300912e-02
    2.weight     : 2.965655e-01
    2.bias       : 1.585917e-02
    4.weight     : 2.461078e-01
    4.bias       : 9.523852e-03


### Lecture du resultat : le reseau de reference

Le reseau de reference est un MLP tanh de **2 couches cachees de 16 neurones** (3 matrices de poids, 337 parametres), entraine 200 pas de SGD a lr=0.05 avec la loss `BCEWithLogitsLoss`. Les valeurs qu il affiche sont la **grille de comparaison** des trois controles qui suivent.

- La **perte affichee** descend de 0.702464 a 0.108086.
- La **metrique de la tache** - la log_loss de scikit-learn, calculee sur les probabilites - vaut 0.107626, et l exactitude atteint 0.9900 (396/400).
- Les **gradients du premier pas** sont enfin lisibles, tenseur par tenseur : 6.563111e-02 pour `0.weight`, 1.300912e-02 pour `0.bias`, 2.965655e-01 pour `2.weight`, 1.585917e-02 pour `2.bias`, 2.461078e-01 pour `4.weight`, 9.523852e-03 pour `4.bias`.

Trois lectures. **La perte n est pas la metrique** : 0.108086 d un cote, 0.107626 de l autre - l ecart est faible ici, mais c est la log_loss qu on compare d un modele a l autre, parce que c est elle qui penalise la confiance excessive. **La repartition des gradients** est le point a retenir : la premiere couche recoit 6.563111e-02, la derniere 2.461078e-01 - le signal circule dans les deux sens, rien ne s effondre. **C est cette repartition qui sert de reference** : les checks 3 et 5 qui suivent la font bouger, et l on verra qu elle est fragile.

In [10]:
# Check 5 - initialisation : que se passe-t-il si tous les poids sont nuls ?
print("=== Check 5 - initialisation nulle (tous les poids a zero) ===")
z0 = mlp(init="zeros")
pertes_z, grads_z = entrainer(z0, nn.BCEWithLogitsLoss(), pas=200, lr=0.05)
ll_z, acc_z = vraies_metriques(z0)
logits_z = z0(X_t).detach().numpy().ravel()
for n, v in grads_z.items():
    print(f"  gradient {n:12s} : {v:.10e}")
print(f"  perte affichee      : {pertes_z[0]:.6f} -> {pertes_z[-1]:.6f} (variation {abs(pertes_z[0] - pertes_z[-1]):.3e} sur 200 pas)")
print(f"  log_loss (tache)    : {ll_z:.6f}")
print(f"  exactitude          : {acc_z:.4f} ({int(round(acc_z * len(y)))}/400)")
print(f"  classe majoritaire  : {np.bincount(y).max()}/400 (exactitude d un modele constant)")
print(f"  logits distincts    : {len(np.unique(np.round(logits_z, 6)))}")

=== Check 5 - initialisation nulle (tous les poids a zero) ===
  gradient 0.weight     : 0.0000000000e+00
  gradient 0.bias       : 0.0000000000e+00
  gradient 2.weight     : 0.0000000000e+00
  gradient 2.bias       : 0.0000000000e+00
  gradient 4.weight     : 0.0000000000e+00
  gradient 4.bias       : 7.4999998324e-03
  perte affichee      : 0.693147 -> 0.693035 (variation 1.118e-04 sur 200 pas)
  log_loss (tache)    : 0.693035
  exactitude          : 0.5075 (203/400)
  classe majoritaire  : 203/400 (exactitude d un modele constant)
  logits distincts    : 1


### Lecture du resultat : l initialisation cassee

Toutes les normes de gradient de **poids** sont exactement nulles : 0.0000000000e+00 pour `0.weight`, `2.weight` et `4.weight`, et 0.0000000000e+00 pour `0.bias` et `2.bias`. Une seule survit : `4.bias` a 7.4999998324e-03 - le biais de sortie, celui dont aucun poids en amont ne peut multiplier la contribution.

La consequence est mecanique. Avec des poids nuls, la sortie du reseau ne depend plus des entrees : les 400 echantillons recoivent **le meme logit** (`logits distincts : 1`). Le reseau est donc un modele constant, et son exactitude - 0.5075, soit 203/400 - est exactement celle de la classe majoritaire. La perte affichee ne bouge presque pas : 0.693147 -> 0.693035, une variation de 1.118e-04 sur 200 pas. Elle reste collee a la perte de l indifference.

**Pourquoi le gradient est-il exactement nul ?** La derivee de la perte par rapport a un poids est le produit d un terme d erreur (non nul ici) par l activation qui alimente ce poids (nulle). Tant que l activation reste nulle, aucun gradient ne peut naitre : c est un etat stationnaire, pas un minimum. Aucun reglage du learning rate ne peut en sortir - diviser un gradient nul par un pas plus grand redonne zero. La parade est une initialisation non degenerate (Kaiming, comme le `default` de PyTorch), et c est precisement ce que le check 5 doit verifier avant tout autre chose.

In [11]:
# Check 3 - gradient : le signal s affaiblit-il en profondeur ?
print("=== Check 3 - profil du gradient selon la profondeur ===")
for depth in (2, 10):
    m = mlp(depth=depth)
    opt = torch.optim.SGD(m.parameters(), lr=0.05)
    opt.zero_grad()
    perte = nn.BCEWithLogitsLoss()(m(X_t), y_t)
    perte.backward()
    poids = {n: float(q.grad.norm())
             for n, q in m.named_parameters() if q.grad is not None and n.endswith("weight")}
    valeurs = list(poids.values())
    print(f"  profondeur {depth:2d} : 1re couche = {valeurs[0]:.6e} | derniere = {valeurs[-1]:.6e} | rapport = {valeurs[-1] / valeurs[0]:.2f}")

print("  -- le reseau profond converge-t-il vraiment ? --")
for pas in (200, 1000, 5000):
    m = mlp(depth=10)
    pertes, _ = entrainer(m, nn.BCEWithLogitsLoss(), pas=pas, lr=0.05)
    ll, acc = vraies_metriques(m)
    print(f"  pas={pas:5d} : perte = {pertes[-1]:.6f} | log_loss = {ll:.6f} | exactitude = {acc:.4f}")

=== Check 3 - profil du gradient selon la profondeur ===
  profondeur  2 : 1re couche = 6.563111e-02 | derniere = 2.461078e-01 | rapport = 3.75
  profondeur 10 : 1re couche = 6.917583e-04 | derniere = 2.535290e-02 | rapport = 36.65
  -- le reseau profond converge-t-il vraiment ? --


  pas=  200 : perte = 0.689598 | log_loss = 0.689565 | exactitude = 0.5075


  pas= 1000 : perte = 0.016014 | log_loss = 0.015893 | exactitude = 0.9925


  pas= 5000 : perte = 0.000572 | log_loss = 0.000572 | exactitude = 1.0000


### Lecture du resultat : le signal qui s affaiblit

Les deux profils de gradient ne se ressemblent pas. A **profondeur 2**, la premiere couche recoit 6.563111e-02 et la derniere 2.461078e-01, un rapport de 3.75. A **profondeur 10**, la premiere couche recoit 6.917583e-04 pour 2.535290e-02 sur la derniere, un rapport de 36.65 : le signal est 36.65 fois plus faible a l entree qu a la sortie. C est le **check 3** qui le dit - et il le dit sur la repartition, pas sur la perte.

Mais l instrument doit etre lu avec la deuxieme moitie de la sortie. Le reseau profond **converge**, si on lui laisse le budget : 200 pas donnent 0.689598 (exactitude 0.5075, c est-a-dire le hasard), 1000 pas donnent 0.016014 (exactitude 0.9925), 5000 pas donnent 0.000572 (exactitude 1.0000).

La lecon est double, et c est elle qui fait la difference entre un diagnostic et une superstition. **Un gradient faible n est pas un gradient nul** : la premiere couche recoit bien un signal, il est seulement 36.65 fois plus petit, et un pas plus grand le remonte. **Un verdict « stagne » a 200 pas n est pas un verdict « casse »** : le reseau n est pas casse, il est lent. Inversement, conclure « il converge » parce qu il converge a 5000 pas serait faux en operation : a 1000 fois le budget nominal, on ne rattrape pas une architecture qui ne sinalise plus. Ce que l instrument doit dire, c est donc « signal affaibli, budget a revoir », pas « mort ».

In [12]:
# Check 2 - la perte affichee dit-elle la verite sur la metrique de la tache ?
print("=== Check 2 - perte affichee vs metrique de la tache ===")


def mse_sur_proba(pred, cible):
    return ((torch.sigmoid(pred) - cible) ** 2).mean()


for nom, perte_fn in (("BCEWithLogits", nn.BCEWithLogitsLoss()), ("MSE(sigmoid)", mse_sur_proba)):
    torch.manual_seed(0)
    m = nn.Sequential(nn.Linear(2, 16), nn.Tanh(), nn.Linear(16, 1))
    pertes, _ = entrainer(m, perte_fn, pas=400, lr=0.05)
    z = m(X_t).detach().numpy().ravel()
    proba = 1.0 / (1.0 + np.exp(-z))
    ll = float(log_loss(y, proba))
    sat = int(((proba < 0.01) | (proba > 0.99)).sum())
    acc = float(((z > 0).astype(int) == y).mean())
    print(f"  {nom:14s} : perte affichee = {pertes[-1]:.6f} | log_loss = {ll:.6f} | exactitude = {acc:.4f}")
    print(f"  {'':14s}   proba dans [{proba.min():.6f}, {proba.max():.6f}] | saturations (p<0.01 ou p>0.99) = {sat}/400")

=== Check 2 - perte affichee vs metrique de la tache ===


  BCEWithLogits  : perte affichee = 0.102390 | log_loss = 0.102249 | exactitude = 0.9900
                   proba dans [0.000770, 0.999417] | saturations (p<0.01 ou p>0.99) = 169/400


  MSE(sigmoid)   : perte affichee = 0.047013 | log_loss = 0.193940 | exactitude = 0.9900
                   proba dans [0.011777, 0.990671] | saturations (p<0.01 ou p>0.99) = 2/400


### Lecture du resultat : la perte qui ment

Les deux lignes se contredisent, et c est exactement le but de ce check. Entrainee avec `BCEWithLogits`, la perte affichee vaut 0.102390 et la log_loss 0.102249 - les deux nombres concordent, comme il se doit quand la loss optimise la metrique. Entrainee avec `MSE(sigmoid)`, la perte affichee vaut **0.047013**, donc une perte affichee **plus faible**, mais la log_loss vaut **0.193940**, donc une metrique presque deux fois plus mauvaise (0.193940 contre 0.102249).

Les deux modeles ont pourtant la meme exactitude : 0.9900. Ce qui les distingue est la **confiance**. Avec `BCEWithLogits`, les probabilites s etalent de 0.000770 a 0.999417 et 169 des 400 predictions sont saturees (p < 0.01 ou p > 0.99) : le modele tranche. Avec `MSE(sigmoid)`, les probabilites restent entre 0.011777 et 0.990671 et il n y a que 2 saturations sur 400 : le modele hesite.

**Pourquoi ?** La MSE appliquee a la probabilite fait passer le gradient par la derivee de la sigmoid, qui s annule quand la proba s approche de 0 ou de 1 : le modele ne peut plus pousser ses predictions vers les extremes, il les laisse au milieu. Sur une metrique qui penalise l indifference, ces predictions molles coutent cher - d ou la log_loss a 0.193940 alors que la loss affichee affiche 0.047013.

La regle qui en decoule : **on juge une loss sur la metrique de la tache, jamais sur la valeur qu elle affiche**. Une perte qui descend est un indice, pas une preuve ; `log_loss` (binaire) ou `cross_entropy` (multi-classes) est le juge, `BCEWithLogitsLoss` est le candidat qui le respecte.

In [13]:
def diagnostic_reseau(modele, perte, lr=0.05, pas=200):
    """Les 5 controles, version reseau. Renvoie la liste des messages."""
    msgs = []
    # check 1 - data pipeline
    msgs += diagnostic_data(X, y)
    # check 5 - initialisation : une matrice de poids entierement nulle ?
    nuls = [n for n, p in modele.named_parameters()
            if p.dim() >= 2 and float(p.detach().abs().max()) == 0.0]
    if nuls:
        msgs.append(f"KO - init : {', '.join(nuls)} est entierement nulle")
    else:
        msgs.append("OK - init : aucun poids entierement nul")
    # check 3 - gradient : NaN, gradient nul, repartition couche par couche
    opt = torch.optim.SGD(modele.parameters(), lr=lr)
    opt.zero_grad()
    p = perte(modele(X_t), y_t)
    p.backward()
    normes = {n: float(q.grad.norm())
              for n, q in modele.named_parameters() if q.grad is not None}
    poids = [v for n, v in normes.items() if n.endswith("weight")]
    if any(np.isnan(v) for v in normes.values()):
        msgs.append("KO - gradient : NaN dans les gradients")
    elif all(v == 0.0 for v in poids):
        msgs.append("KO - gradient : toutes les normes de gradient de poids sont nulles")
    else:
        msgs.append(f"OK - gradient : 1re couche = {poids[0]:.3e} | derniere = {poids[-1]:.3e}")
        if poids[0] < 1e-3:
            msgs.append(f"WARN - gradient : 1re couche < 1e-3 (le signal s affaiblit en profondeur)")
    # check 4 - learning rate, mesure sur la trajectoire
    pertes, _ = entrainer(modele, perte, pas=pas, lr=lr)
    saut = max(abs(b - a) for a, b in zip(pertes, pertes[1:]))
    if pertes[-1] > 1.0:
        msgs.append(f"KO - lr={lr} : explose (perte finale {pertes[-1]:.2f} > 1)")
    elif saut > 0.1:
        msgs.append(f"KO - lr={lr} : oscille (saut max {saut:.2f} entre deux passes)")
    elif pertes[-1] > 0.68:
        msgs.append(f"KO - lr={lr} : stagne (perte finale {pertes[-1]:.4f}, au voisinage du hasard)")
    else:
        msgs.append(f"OK - lr={lr} : converge (perte finale {pertes[-1]:.4f})")
    # check 2 - la metrique de la tache, comparee a une loss adaptee de reference
    ll, acc = vraies_metriques(modele)
    profondeur = sum(1 for couche in modele if isinstance(couche, nn.Linear)) - 1
    temoin = mlp(depth=profondeur)
    entrainer(temoin, nn.BCEWithLogitsLoss(), pas=pas, lr=lr)
    ll_ref, _ = vraies_metriques(temoin)
    if ll > 0.5:
        msgs.append(f"KO - metrique : log_loss = {ll:.4f} > 0.5, la metrique de la tache est degradee")
    elif ll > 1.5 * ll_ref:
        msgs.append(f"KO - metrique : log_loss = {ll:.4f} = {ll / ll_ref:.1f} x la reference ({ll_ref:.4f}) alors que la perte affichee vaut {pertes[-1]:.4f} - loss mal adaptee")
    else:
        msgs.append(f"OK - metrique : log_loss = {ll:.4f} (reference {ll_ref:.4f}) | exactitude = {acc:.4f} | perte affichee = {pertes[-1]:.4f}")
    return msgs


print("=== Instrument : les 5 controles sur 4 configurations ===")
configurations = [
    ("sain (kaiming, profondeur 2)", mlp(depth=2), nn.BCEWithLogitsLoss()),
    ("initialisation nulle", mlp(depth=2, init="zeros"), nn.BCEWithLogitsLoss()),
    ("profondeur 10 (tanh)", mlp(depth=10), nn.BCEWithLogitsLoss()),
    ("loss inadaptee (MSE sur sigmoid)", mlp(depth=2), mse_sur_proba),
]
for nom, modele, perte_fn in configurations:
    print(f"\n-- {nom}")
    for msg in diagnostic_reseau(modele, perte_fn, lr=0.05, pas=200):
        print(f"   {msg}")

=== Instrument : les 5 controles sur 4 configurations ===

-- sain (kaiming, profondeur 2)


   OK - X.shape=(400, 2), X.std=0.984, NaN=0
   OK - init : aucun poids entierement nul
   OK - gradient : 1re couche = 6.563e-02 | derniere = 2.461e-01
   OK - lr=0.05 : converge (perte finale 0.1081)
   OK - metrique : log_loss = 0.1076 (reference 0.1076) | exactitude = 0.9900 | perte affichee = 0.1081

-- initialisation nulle


   OK - X.shape=(400, 2), X.std=0.984, NaN=0
   KO - init : 0.weight, 2.weight, 4.weight est entierement nulle
   KO - gradient : toutes les normes de gradient de poids sont nulles
   KO - lr=0.05 : stagne (perte finale 0.6930, au voisinage du hasard)
   KO - metrique : log_loss = 0.6930 > 0.5, la metrique de la tache est degradee

-- profondeur 10 (tanh)


   OK - X.shape=(400, 2), X.std=0.984, NaN=0
   OK - init : aucun poids entierement nul
   OK - gradient : 1re couche = 6.918e-04 | derniere = 2.535e-02
   WARN - gradient : 1re couche < 1e-3 (le signal s affaiblit en profondeur)
   KO - lr=0.05 : stagne (perte finale 0.6896, au voisinage du hasard)
   KO - metrique : log_loss = 0.6896 > 0.5, la metrique de la tache est degradee

-- loss inadaptee (MSE sur sigmoid)


   OK - X.shape=(400, 2), X.std=0.984, NaN=0
   OK - init : aucun poids entierement nul
   OK - gradient : 1re couche = 3.278e-02 | derniere = 1.230e-01
   OK - lr=0.05 : converge (perte finale 0.0626)
   KO - metrique : log_loss = 0.2546 = 2.4 x la reference (0.1076) alors que la perte affichee vaut 0.0626 - loss mal adaptee


### Lecture du resultat : les 5 controles sur 4 configurations

L instrument repond sur les quatre reseaux, et les quatre verdicts ne se ressemblent pas.

- **Sain** - cinq OK. Init OK, gradient OK (1re couche 6.563e-02, derniere 2.461e-01), lr=0.05 converge (0.1081), metrique OK (log_loss 0.1076, reference 0.1076, exactitude 0.9900).
- **Initialisation nulle** - le check 5 tombe (`KO - init : 0.weight, 2.weight, 4.weight est entierement nulle`), et le check 3 tombe avec lui (`KO - gradient : toutes les normes de gradient de poids sont nulles`). Le lr est alors juge `KO - lr=0.05 : stagne (perte finale 0.6930, au voisinage du hasard)` et la metrique `KO - log_loss = 0.6930 > 0.5`. Quatre verdicts KO, une seule cause.
- **Profondeur 10** - l init est OK, le gradient est en `WARN` (1re couche 6.918e-04 < 1e-3, derniere 2.535e-02), le lr est KO (stagne, 0.6896) et la metrique KO (0.6896 > 0.5). L instrument signale un **budget**, pas une panne - c est l autopsie de la cellule precedente qui tranche.
- **Loss inadaptee** - init OK, gradient OK (3.278e-02 / 1.230e-01), lr OK (converge, 0.0626), et **seulement** la metrique tombe : `KO - metrique : log_loss = 0.2546 = 2.4 x la reference (0.1076) alors que la perte affichee vaut 0.0626 - loss mal adaptee`.

Ce dernier cas est le plus instructif, parce qu il ne se laisse pas voir autrement. Ni l init, ni le gradient, ni le learning rate n ont rien a dire : le run est **propre**, et c est pourtant la metrique qui est mauvaise. Un diagnostic qui s arretait au graphique de la perte aurait conclu « tout va bien ».

Le message cle de l instrument est donc celui de sa derniere ligne : il compare la metrique a une **reference** entrainee avec la loss adaptee, et c est ce rapport - et non la valeur absolue - qui denonce une loss mal adaptee. Une valeur absolue dirait « 0.2546, c est correct » ; le rapport dit « 2.4 fois la reference, c est faux ».

## 6. Conclusion

Quand un entrainement ne baisse pas, **les 5 checks sont un parcours**, pas une check-list - on avance dans l ordre, et chaque check KO doit vous orienter vers une hypothese precise.

Les patterns les plus frequents -

- **lr trop petit** - perte stagne au niveau du hasard. Solution - augmenter `eta0` d un facteur 10.
- **lr trop grand** - perte oscille puis explose au-dela de 1. Solution - reduire `eta0` ou ajouter du gradient clipping.
- **data pipeline** - NaN/Inf dans X ou y. Solution - verifier la source, `np.isnan(X).any()`.
- **loss incorrecte** - MSE sur classification. Solution - `log_loss` (binaire) ou `categorical_crossentropy` (multi-classes).
- **mauvaise initialisation** - modele constant. Solution - Glorot/Xavier ou le default framework.
- **gradient qui disparait** - la premiere couche ne recoit presque plus de signal. Solution - moins de couches, une activation moins saturante, ou un pas plus grand.

**Ce que la section 5 ajoute.** Sur le modele lineaire, deux checks sur cinq sont observables ; sur le reseau, les cinq le sont, et chacun change de nature :

- **check 5 (initialisation)** n est pas une hypothese parmi d autres : a poids tous nuls, les gradients des poids sont **exactement nuls** et le reseau ne peut pas s echoes. C est une panne d initialisation, pas un lr mal choisi - et aucun réglage du pas ne la repare.
- **check 3 (gradient)** ne se lit pas sur la perte mais sur la **repartition couche par couche** : une premiere couche a 6.917583e-04 face a 2.535290e-02 sur la derniere, c est un signal qui s affaiblit. Le reseau converge quand meme si on lui laisse assez de budget - un verdict « stagne » a 200 pas n est pas un verdict « casse ».
- **check 2 (loss)** se juge sur la **metrique de la tache**, pas sur la perte affichee : une loss mal adaptee peut produire une perte affichee meilleure et une metrique deux fois plus mauvaise.

**Suite naturelle** - ce carnet ferme la sous-serie 03-Deep-Learning. La [feuille de route de la serie](README.md) donne les parcours suivants ; `02-ML-Cours/2.13-Analyse-Erreurs.ipynb` est le pendant en ML classique.